# 1D Quantum Tunneling — Rectangular Barrier

**Phase 5** of `TDSE_Solver_Plan.md`: a wavepacket incident on a rectangular
potential barrier, `V(x) = V0` for `|x| <= a/2` and `0` elsewhere. The measured
transmission probability (how much of `|psi|^2` ends up past the barrier at long
time) is compared against the exact textbook result for a monochromatic plane
wave of the same energy,
`potentials.rectangular_barrier_transmission(E, V0, a)`
(covering both the sub-barrier tunneling regime and the over-barrier regime,
which shows resonant perfect transmission).

A real Gaussian wavepacket isn't monochromatic, though -- it has energy spread
`~1/(8*sigma0^2)` around its central energy `E0=k0^2/2` (from the momentum-space
width `sigma_p=1/(2*sigma0)` of `grid.gaussian_wavepacket`). Since `T(E)` is
highly curved (especially in the deep-tunneling regime, where it's exponentially
sensitive to `E`), the wavepacket-averaged transmission probability will differ
from the pointwise plane-wave value by more than machine precision -- a real,
expected physical effect, not a bug. The energies and `sigma0` below were chosen
(by direct experimentation) so this systematic stays under ~0.01 in absolute
transmission probability across the whole sweep.

In [1]:
import sys
sys.path.insert(0, r".")
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import imageio_ffmpeg
from pathlib import Path

import grid as g
import propagator as prop
import potentials as pot
import observables as obs

matplotlib.rcParams['animation.ffmpeg_path'] = imageio_ffmpeg.get_ffmpeg_exe()
MEDIA_DIR = Path('media')
MEDIA_DIR.mkdir(exist_ok=True)


## Transmission probability vs. incident energy

Barrier: `V0=1`, `a=2` (atomic units). Domain large enough (`L=240`) that neither
the reflected nor transmitted lobe reaches the domain edge within the run. The
wavepacket starts at `x0=-50` with `sigma0=8` (narrow momentum spread); the run
time for each energy is scaled as `(|x0|+D_sep)/k0` so every energy's packet
travels the same physical distance before being measured, regardless of its
speed -- keeping the same domain adequate for the whole sweep.

In [2]:
V0, a = 1.0, 2.0
L, N = 240.0, 2048
grid1d = g.make_grid((L,), (N,), boundary='periodic')
V_barrier = pot.rectangular_barrier(grid1d, V0, -a / 2, a / 2)

x0 = -50.0
sigma0 = 8.0
D_sep = 35.0
dt = 0.01

energies = np.array([0.3, 0.5, 0.7, 0.9, 1.1, 1.5, 2.0])
T_measured = []
k2_barrier = prop.kinetic_eigenvalues(grid1d)

for E in energies:
    k0 = np.sqrt(2 * E)
    psi = g.gaussian_wavepacket(grid1d, center=x0, sigma=sigma0, k0=k0)
    T_total = (abs(x0) + D_sep) / k0
    n_steps = round(T_total / dt)
    for _ in range(n_steps):
        psi = prop.strang_step(psi, grid1d, V_barrier, dt, k2=k2_barrier)
    P_trans = obs.region_probability(psi, grid1d, axis=0, x_min=a / 2)
    P_refl = obs.region_probability(psi, grid1d, axis=0, x_max=-a / 2)
    T_measured.append(P_trans)
    print(f"E={E:.2f}  P_trans={P_trans:.4f}  P_refl={P_refl:.4f}  sum={P_trans + P_refl:.4f}")

T_measured = np.array(T_measured)
T_analytic_pts = pot.rectangular_barrier_transmission(energies, V0, a)


E=0.30  P_trans=0.0307  P_refl=0.9693  sum=0.9999


E=0.50  P_trans=0.0737  P_refl=0.9262  sum=1.0000


E=0.70  P_trans=0.1478  P_refl=0.8522  sum=1.0000


E=0.90  P_trans=0.2642  P_refl=0.7357  sum=1.0000


E=1.10  P_trans=0.4244  P_refl=0.5755  sum=0.9999


E=1.50  P_trans=0.7751  P_refl=0.2248  sum=0.9999


E=2.00  P_trans=0.9815  P_refl=0.0184  sum=0.9999


In [3]:
# Smooth analytic reference curve (many more points than the simulated
# energies) plus the simulated (wavepacket-averaged) points
E_smooth = np.linspace(0.05, 2.4, 400)
T_smooth = pot.rectangular_barrier_transmission(E_smooth, V0, a)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(E_smooth, T_smooth, '-', color='C0', label='analytic plane-wave T(E)')
ax.plot(energies, T_analytic_pts, 'o', color='C0', ms=5)
ax.plot(energies, T_measured, 's', color='C3', ms=7, label='simulated (wavepacket-averaged)')
ax.axvline(V0, color='gray', linestyle=':', linewidth=1, label='V0')
ax.set_xlabel('Incident energy E')
ax.set_ylabel('Transmission probability T')
ax.set_title(f'Rectangular barrier tunneling: V0={V0}, a={a}')
ax.legend()
ax.set_ylim(0, 1.05)
fig.tight_layout()
fig.savefig(MEDIA_DIR / 'transmission_vs_energy.png', dpi=150)
plt.show()


C:\Users\Hasan's Laptop\AppData\Local\Temp\ipykernel_30792\1513847257.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
abs_err = np.abs(T_measured - T_analytic_pts)
rel_err = abs_err / T_analytic_pts
print(f"{'E':>6} {'T_measured':>11} {'T_analytic':>11} {'abs_err':>9} {'rel_err':>9}")
for E, tm, ta, ae, re in zip(energies, T_measured, T_analytic_pts, abs_err, rel_err):
    print(f"{E:6.2f} {tm:11.4f} {ta:11.4f} {ae:9.4f} {re:9.3f}")

max_abs_err = np.max(abs_err)
print(f"\nmax absolute error = {max_abs_err:.4f}")
assert max_abs_err < 0.015, "transmission probability disagrees with the analytic curve by more than the expected wavepacket energy-spread systematic"
print("PASS: measured transmission probability tracks the analytic curve within the expected wavepacket energy-spread systematic")


     E  T_measured  T_analytic   abs_err   rel_err
  0.30      0.0307      0.0292    0.0014     0.049
  0.50      0.0737      0.0707    0.0031     0.044
  0.70      0.1478      0.1426    0.0052     0.037
  0.90      0.2642      0.2576    0.0066     0.026
  1.10      0.4244      0.4198    0.0046     0.011
  1.50      0.7751      0.7839    0.0089     0.011
  2.00      0.9815      0.9883    0.0068     0.007

max absolute error = 0.0089
PASS: measured transmission probability tracks the analytic curve within the expected wavepacket energy-spread systematic


## Animation: E=0.5 tunneling event

For the animation, a complex absorbing potential (CAP, `potentials.
absorbing_boundary`) is added at both domain edges so the reflected and
transmitted lobes fade away cleanly after separating, rather than wrapping
around the periodic domain -- purely a visualization convenience; it plays no
role in the transmission-probability measurement above.

In [5]:
E_hero = 0.5
k0_hero = np.sqrt(2 * E_hero)
psi0_hero = g.gaussian_wavepacket(grid1d, center=x0, sigma=sigma0, k0=k0_hero)
W = pot.absorbing_boundary(grid1d, width=15.0, eta=15.0, order=3)
V_hero = V_barrier - 1j * W

T_hero = 100.0
n_steps_hero = round(T_hero / dt)
target_frames = 220
stride = max(1, n_steps_hero // target_frames)

frames = []
psi = psi0_hero.copy()
t = 0.0
for step in range(n_steps_hero):
    psi = prop.strang_step(psi, grid1d, V_hero, dt, k2=k2_barrier)
    t += dt
    if step % stride == 0:
        frames.append((t, obs.probability_density(psi).copy()))
print(f"{len(frames)} frames recorded")


223 frames recorded


In [6]:
x = grid1d.axes[0]
fig, ax = plt.subplots(figsize=(9, 4.5))
line, = ax.plot(x, frames[0][1], color='C0')
barrier_fill = ax.axvspan(-a / 2, a / 2, color='gray', alpha=0.3, label='barrier')
ax.set_xlim(-80, 80)
ymax = max(np.max(f[1]) for f in frames[:len(frames) // 3])
ax.set_ylim(0, ymax * 1.15)
ax.set_xlabel('x')
ax.set_ylabel('probability density |psi|^2')
title = ax.set_title(f'Tunneling, E={E_hero} (V0={V0}): t=0.00')
ax.legend(loc='upper right')

def update(i):
    t_i, density = frames[i]
    line.set_ydata(density)
    title.set_text(f'Tunneling, E={E_hero} (V0={V0}): t={t_i:.2f}')
    return line, title

ani = animation.FuncAnimation(fig, update, frames=len(frames), interval=50, blit=False)
ani.save(MEDIA_DIR / 'tunneling_1d.mp4', writer='ffmpeg', fps=24, dpi=120)
plt.close(fig)
print('saved media/tunneling_1d.mp4')


saved media/tunneling_1d.mp4
